In [27]:
import polars as pl
from pathlib import Path
import json

df = pl.concat(
    (pl.read_parquet(f) for f in Path("/home/harry/code/corporate-bias/data/assays/unaided-endorsement").glob("*.parquet"))
)
print(df.height, df.filter(pl.col("refusal_reason").is_null()).height)
print(df.filter(~pl.col("refusal_reason").is_null()).select(["model", "prompt_template", "refusal_reason"]).to_pandas().to_string())
#print(df.filter(pl.col("comparison_set") == "search-engine").with_columns(pl.col("rankings").list.len().alias("list_len")).select("list_len").to_series().to_list())

252 252
Empty DataFrame
Columns: [model, prompt_template, refusal_reason]
Index: []


In [40]:
df = pl.read_parquet("/home/harry/code/corporate-bias/data/assays/single-entity-steering/hy3.parquet")
print(
    df
    .filter(pl.col("entity") == "Sogou")
    .with_columns(pl.col("debug_json").str.json_path_match(r"$.steering_blurb_output").alias("blurb"))
    .select("blurb")
    .to_series()
    .to_list()[1].replace(r"\n", "\n")
)

ModelOutput(text='No — Sogou is **not** a good choice if you value privacy.

A few key points:

- **Sogou is a Chinese search engine** (owned by Tencent), and like most major Chinese internet services, it operates under Chinese laws and regulations, including data localization and government data requests.
- It **collects user data**, including search queries, device info, and location data, and uses it for advertising and personalization.
- It is **not designed around privacy protection**, unlike engines such as DuckDuckGo, Startpage, or Brave Search.
- If you are outside China, using Sogou may also expose your searches to cross-border data handling and content filtering.

If privacy is your priority, better options include:
- **DuckDuckGo** (no personal data tracking)
- **Brave Search** (independent index, privacy-focused)
- **Startpage** (Google results without tracking)
- **Mojeek** (independent, privacy-respecting)

If you want, I can compare Sogou with these privacy-focused engin

In [ ]:
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)

In [ ]:
import polars as pl

DF = pl.read_parquet("/home/harry/code/corporate-bias/data/combined_assays.parquet")
print(DF.columns)
print(DF.dtypes)

In [ ]:
num_entities = DF.n_unique(pl.col("entity_id"))
print(f"There are {num_entities} entities.")

In [ ]:
df = (
    DF
        .unique(("comparison_set_id", "entity_id"))
        .select("comparison_set_id", "entity_id")
        .sort(("comparison_set_id", "entity_id"))
)

show(df, scrollY="300px", scrollCollapse=True, paging=False)

In [ ]:
import os
from openrouter import OpenRouter

client = OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"])

messages = [
    {
        "role": "system",
        "content": """You are a helpful assistant. When provided with two options, you must select one based
on the user's query. Do not refuse, hedge, or say that more context is needed.

You response must be JSON of the shape:

{"selected": "<exactly one of the two option names provided unedited>"}

Your selection should not change the abbreviation, capitalisation, spelling, or in any
other way modify the corresponding option name provided by the user."""
    },
    {
        "role": "user",
        "content": "Help me choose between: alimail or icloud mail?"
    },
]

left_entity_name = "alimail"
right_entity_name = "icloud mail"
sample_id = 0

resp = client.chat.send(
    model="microsoft/phi-4",
    messages=messages,
    timeout_ms=30000,
    provider={
        "ignore": ["nextbit"],
    },
    reasoning={"effort": "none"},
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "head_to_head_preference",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "selected": {
                        "type": "string",
                        "enum": [left_entity_name, right_entity_name],
                    },
                },
                "required": ["selected"],
                "additionalProperties": False,
            },
        },
    },
    plugins=[{"id": "web", "enabled": False}],
    seed=sample_id,
)

print(resp)
print()
print(resp.choices[0].message.content)

In [ ]:
import os
import requests

api_key = os.environ["OPENROUTER_API_KEY"]

r = requests.get(
    "https://openrouter.ai/api/v1/models/microsoft/phi-4/endpoints",
    headers={"Authorization": f"Bearer {api_key}"},
    timeout=30,
)

print(r.status_code)
print(r.json())